# 第2章 机器学习基础 · 参考答案



本册为依据课程材料新编的训练题，不是教师公布的考试原题。

## 材料依据
- [[课程/深度学习/PPT/Chapter2/Chapter2.md]]
- [[课程/深度学习/课堂笔记/2026-09-02-第2节-机器学习基础概念与模型构建教学.md]]
- [[课程/深度学习/课堂笔记/2026-09-07-第3节-机器学习基础与NumPy实操讲座.md]]
- [[课程/深度学习/演示代码/Chapter2.py]]
- [[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]]
- [[课程/深度学习/PPT/2026深度学习第一次上机考试规则与命令表v1.1 (1)/2026深度学习第一次上机考试规则与命令表v1.1 (1).md]]


## 使用方式

先运行环境单元，再逐题阅读、实现和自检。教学题允许提示与接口骨架；遇到“尚未完成”不代表通过。每题的代码需写在该题作答单元，答案在独立文件夹。学习时间不受 30 分钟限制；综合作业扩展题可分次完成。


In [ ]:
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from pathlib import Path

# CPU 默认即可；不更改全局默认设备，也不依赖 CUDA、网络或额外数据文件。
torch.set_num_threads(1)
np.set_printoptions(precision=6, suppress=True)
torch.manual_seed(2026)
print("NumPy", np.__version__, "PyTorch", torch.__version__)


## L2-01 数组、切片与归约的实际数值

**来源与性质**：[[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]] C2-06—C2-08；[[课程/深度学习/课堂笔记/2026-09-07-第3节-机器学习基础与NumPy实操讲座.md]]“NumPy常用操作”；[[课程/深度学习/演示代码/Chapter2.py]]前半段。合并改编，扩充轴方向检查。


实现无参数函数 `array_lab()`，返回字典：
- `ones`：`(3,2)`全1数组（可以用zeros加1）；`zeros`：`(2,3)`全0数组；`grid`：`[-1,1]`的5点等距网格；`even`：0到10不含10、步长2的序列。
- `C_value`：`C=[[1,2],[-1,-1],[0,2]]` 的 `[1,0]`；`C_head`：前两行。
- `D`：0—15按行重排的 `(4,4)` 数组；`middle`：`D[1:3,1:3]`；`transpose`：D的转置。
- `scaled_sine`：`10*sin(D)`；`rounded`：前者四舍六入五成双到整数；`row_mean`：前者每行均值；`column_max`：前者每列最大值。

打印主要结果，并说明切片停止端点不包含。无随机性。教学扩展：`reshape` 与 `np.around`（都在课件／演示中，但未逐项列入考试命令表）。


<details><summary>提示（先尝试再展开）</summary>

先用 `np.arange(16)` 定位每个元素，再检查中间块；按行归约是axis=1，按列归约是axis=0。

</details>


In [ ]:
import numpy as np
def array_lab():
    C=np.array([[1.,2.],[-1.,-1.],[0.,2.]])
    D=np.arange(16).reshape((4,4))
    S=10*np.sin(D)
    return dict(ones=np.zeros((3,2))+1,zeros=np.zeros((2,3)),
                grid=np.linspace(-1,1,5),even=np.arange(0,10,2),
                C_value=C[1,0],C_head=C[:2,:],D=D,middle=D[1:3,1:3],
                transpose=D.T,scaled_sine=S,rounded=np.around(S),
                row_mean=np.mean(S,axis=1),column_max=np.max(S,axis=0))
array_result=array_lab()
for _key,_value in array_result.items():
    print(_key,':',_value)
print('The stopping index of a slice is excluded.')


### 解析与点评

这些操作不是为了背打印格式，而是理解每个位置的数学含义。最大值与argmax索引是不同对象；row_mean不是整个数组的总均值。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
_r=array_lab()
assert np.array_equal(_r['ones'],[[1,1],[1,1],[1,1]])
assert np.array_equal(_r['zeros'],[[0,0,0],[0,0,0]])
assert np.array_equal(_r['grid'],[-1,-.5,0,.5,1]) and np.array_equal(_r['even'],[0,2,4,6,8])
assert _r['C_value']==-1 and np.array_equal(_r['C_head'],[[1,2],[-1,-1]])
assert np.array_equal(_r['middle'],[[5,6],[9,10]])
assert np.array_equal(_r['transpose'][:,1],[4,5,6,7])
assert np.allclose(_r['scaled_sine'][0],[0,8.414709848,9.092974268,1.411200081],atol=1e-8)
assert np.array_equal(_r['rounded'][0],[0,8,9,1])
assert np.allclose(_r['row_mean'],np.mean(_r['scaled_sine'],axis=1))
assert np.allclose(_r['column_max'],np.max(_r['scaled_sine'],axis=0))

print("自检通过")


## L2-02 矩阵乘法、逐元素乘法与广播陷阱

**来源与性质**：[[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]] C2-09、A-03；[[课程/深度学习/课堂笔记/2026-09-07-第3节-机器学习基础与NumPy实操讲座.md]]第三节及自测；[[课程/深度学习/演示代码/Chapter2.py]] F@v1 与 F@v1T。改编。


实现 `compare_products(F,v,target)`：`F` 为 `(N,d)`、`v` 为 `(d,)`、`target` 为 `(N,)`。返回字典 `flat,column,elementwise,residual,mse`，分别为 `F@v`、`F@v.reshape((d,1))`、`F*v`、**一维逐样本残差**、其MSE。列向量重排属已讲过的教学扩展。

使用 `F=[[1,2,3],[0,-1,2],[2,0,-1],[1,1,1]]`、`v=[1,2,3]`、`target=[13,5,-2,7]`。打印三类乘法及残差的shape，再单独打印 `column-target` 的shape，解释为何后者错误。无随机性。


<details><summary>提示（先尝试再展开）</summary>

`F*v` 会对每一行逐元素加权，但还没把特征维求和；`F@v` 已经完成加权求和。

</details>


In [ ]:
import numpy as np
def compare_products(F,v,target):
    flat=F@v
    column=F@v.reshape((v.size,1))
    residual=flat-target
    return dict(flat=flat,column=column,elementwise=F*v,residual=residual,
                mse=float(np.mean(residual**2)))
_F=np.array([[1.,2.,3.],[0.,-1.,2.],[2.,0.,-1.],[1.,1.,1.]])
_v=np.array([1.,2.,3.]); _t=np.array([13.,5.,-2.,7.])
product_result=compare_products(_F,_v,_t)
for _k,_a in product_result.items():
    print(_k,':',_a)
print('incorrect broadcast shape:',(product_result['column']-_t).shape)


### 解析与点评

两种矩阵乘法的数值按行一致，形状不同。列预测减一维标签会产生N×N残差；若直接求均值就变成了另一道数学问题。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
_r=compare_products(np.array([[1.,2.,3.],[0.,-1.,2.],[2.,0.,-1.],[1.,1.,1.]]),np.array([1.,2.,3.]),np.array([13.,5.,-2.,7.]))
assert np.array_equal(_r['flat'],[14,4,-1,6])
assert _r['column'].shape==(4,1) and np.array_equal(_r['column'][:,0],_r['flat'])
assert np.array_equal(_r['elementwise'][0],[1,4,9])
assert np.array_equal(_r['residual'],[1,-1,1,-1]) and _r['mse']==1
_s=compare_products(np.array([[2.,-1.]]),np.array([3.,4.]),np.array([1.]))
assert _s['mse']==1 and np.array_equal(_s['flat'],[2])

print("自检通过")


## L2-03 可复现采样与高维单位立方体

**来源与性质**：[[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]] C2-05、C2-10；[[课程/深度学习/课堂笔记/2026-09-07-第3节-机器学习基础与NumPy实操讲座.md]]“随机数与可复现性”。改编。


实现 `sample_cube(n,d,seed)`：用局部 `np.random.default_rng(seed)` 生成 `[0,1)` 中的 `(n,d)` 均匀样本，返回 `(X,y)`，其中 `y` 为各行坐标平方的平均数，形状 `(n,)`。`n<=0` 或 `d<=0` 应抛 `ValueError`。

生成 `n=10000,d=10,seed=20261007`，打印各维样本均值、y均值、最小最大坐标。重复同种子应逐元素相同；不同种子应不同。说明二维每轴100格很容易，而10维每轴100格需要多少点。无需真的建立该网格。


<details><summary>提示（先尝试再展开）</summary>

每行是一个点，维数决定列数。连续均匀分布在1上取到端点的概率不是本题判定标准，API区间按[0,1)实现。

</details>


In [ ]:
import numpy as np
def sample_cube(n,d,seed):
    if n<=0 or d<=0:
        raise ValueError('n and d must be positive')
    rng=np.random.default_rng(seed)
    X=rng.uniform(0,1,(n,d))
    return X,np.mean(X**2,axis=1)
cube_X,cube_y=sample_cube(10000,10,20261007)
print('coordinate means:',np.mean(cube_X,axis=0))
print('mean label / min / max:',np.mean(cube_y),cube_X.min(),cube_X.max())
print('10-D grid size:',100**10)


### 解析与点评

10维网格有10²⁰个点，随机抽10000点避免存储完整网格，但不意味着覆盖了所有可能位置。固定种子要同时固定调用顺序和生成尺寸。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
_X,_y=sample_cube(10000,10,20261007)
assert _X.shape==(10000,10) and _y.shape==(10000,)
assert np.all((_X>=0)&(_X<1))
assert np.allclose(_y,np.mean(_X**2,axis=1)) and abs(np.mean(_y)-1/3)<.01
assert np.array_equal(_X,sample_cube(10000,10,20261007)[0])
assert not np.array_equal(_X,sample_cube(10000,10,20261008)[0])
_a,_b=sample_cube(1,1,7)
assert np.isclose(_a[0,0],0.625095466604667) and np.isclose(_b[0],_a[0,0]**2)
for _n,_d in [(0,1),(1,0),(-1,2)]:
    try: sample_cube(_n,_d,1)
    except ValueError: pass
    else: raise AssertionError('非正维数应被拒绝')

print("自检通过")


## L2-04 九点有理函数的二次拟合

**来源与性质**：[[课程/深度学习/PPT/Chapter2/Chapter2.md]]“例1”；[[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]] C2-01；[[课程/深度学习/课堂笔记/2026-09-02-第2节-机器学习基础概念与模型构建教学.md]]“课件例题与代码的相同点、不同点”。改编。


训练横坐标 `[-.92,-.78,-.34,-.33,-.16,.06,.19,.64,.79]`，测试横坐标 `[-.8,-.4,0,.4,.8]`；标签均为 `f(x)=1/(x²+x+1)`。

实现 `poly_matrix(x,p)`：返回升幂列 `[1,x,...,x**p]`；`fit_poly(x,y,p)` 返回最小二乘系数。p为非负整数；支持p=0及单个横坐标。用p=2求 `nine_coef`，保存 `nine_train_rmse,nine_test_rmse,nine_gap_abs,nine_gap_signed`；其中绝对差是课件的泛化差距，有符号差只作额外诊断。打印系数及各指标，画训练点、测试点、目标与拟合曲线，保存 `output/L2-04.png`。

误差按观测标签计算，系数按升幂排列；没有随机性，也不能把这个例子说成Chapter2.py的正弦实验。


<details><summary>提示（先尝试再展开）</summary>

列索引j就是x的j次幂；最小二乘优化的是MSE，但取平方根不改变最优系数。

</details>


In [ ]:
import numpy as np
def poly_matrix(x,p):
    return np.column_stack([x**j for j in range(p+1)])
def poly_ridge(x,y,p,lam):
    if lam < 0:
        raise ValueError('lambda must be nonnegative')
    A=poly_matrix(x,p)
    if lam == 0:
        return np.linalg.lstsq(A,y,rcond=None)[0]
    n=x.size
    B=np.zeros((n+p+1,p+1)); z=np.zeros(n+p+1)
    B[:n,:]=A; z[:n]=y
    for j in range(p+1):
        B[n+j,j]=np.sqrt(n*lam)
    return np.linalg.lstsq(B,z,rcond=None)[0]

import matplotlib.pyplot as plt
from pathlib import Path
def fit_poly(x,y,p):
    return np.linalg.lstsq(poly_matrix(x,p),y,rcond=None)[0]
nine_x=np.array([-.92,-.78,-.34,-.33,-.16,.06,.19,.64,.79])
nine_y=1/(nine_x**2+nine_x+1)
nine_xt=np.array([-.8,-.4,0.,.4,.8])
nine_yt=1/(nine_xt**2+nine_xt+1)

nine_coef=fit_poly(nine_x,nine_y,2)
nine_train_rmse=float(np.sqrt(np.mean((poly_matrix(nine_x,2)@nine_coef-nine_y)**2)))
nine_test_rmse=float(np.sqrt(np.mean((poly_matrix(nine_xt,2)@nine_coef-nine_yt)**2)))
nine_gap_signed=nine_test_rmse-nine_train_rmse
nine_gap_abs=abs(nine_gap_signed)
print('ascending coefficients:',nine_coef)
print('train_RMSE test_RMSE gap_abs gap_signed')
print(nine_train_rmse,nine_test_rmse,nine_gap_abs,nine_gap_signed)
Path('output').mkdir(exist_ok=True)
plt.figure(); _g=np.linspace(-1,1,201)
plt.plot(_g,1/(_g**2+_g+1),'k--',label='target')
plt.plot(_g,poly_matrix(_g,2)@nine_coef,label='degree 2')
plt.plot(nine_x,nine_y,'o',label='train'); plt.plot(nine_xt,nine_yt,'s',label='test')
plt.xlabel('x'); plt.ylabel('y'); plt.legend()
plt.savefig('output/L2-04.png'); plt.close()


### 解析与点评

二次多项式只有3个系数，不能穿过全部九点。有理函数、正弦函数与指数函数是不同目标，误差表不能跨实验抄用。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
from pathlib import Path
assert np.array_equal(poly_matrix(np.array([-1.,0.,2.]),2),[[1,-1,1],[1,0,0],[1,2,4]])
assert np.array_equal(poly_matrix(np.array([2.]),0),[[1]])
assert np.allclose(fit_poly(np.array([-1.,0.,1.]),np.array([2.,1.,6.]),2),[1,2,3])
assert np.allclose(fit_poly(np.array([0.,1.]),np.array([2.,4.]),0),[3])
_x=np.array([-.92,-.78,-.34,-.33,-.16,.06,.19,.64,.79]); _y=1/(_x**2+_x+1)
_A=poly_matrix(_x,2)
assert np.linalg.norm(_A.T@(_A@nine_coef-_y)) < 1e-10
assert .07 < nine_train_rmse < .09 and .07 < nine_test_rmse < .10
assert np.isclose(nine_gap_signed,nine_test_rmse-nine_train_rmse)
assert np.isclose(nine_gap_abs,abs(nine_gap_signed))
assert Path('output/L2-04.png').stat().st_size > 1000

print("自检通过")


## L2-05 划分索引与只用训练集做预处理

**来源与性质**：[[课程/深度学习/PPT/Chapter2/Chapter2.md]]“验证集”；[[课程/深度学习/课堂笔记/2026-09-02-第2节-机器学习基础概念与模型构建教学.md]]第八、九节；[[课程/深度学习/课堂笔记/2026-09-07-第3节-机器学习基础与NumPy实操讲座.md]]第七节。新编防泄漏练习。


实现 `split_and_scale(X,n_train,n_val,seed)`：X为 `(N,d)`，要求三组都非空。生成 `order=np.argsort(rng.uniform(0,1,N))`，前n_train个索引训练，接着n_val个验证，剩余测试。返回字典 `train_idx,val_idx,test_idx,mean,scale,scaled`。

只用训练索引对应行算每列均值与总体标准差（`sqrt(mean((x-mean)**2))`，不除N-1）；零标准差列把scale置1。用这些统计量变换**所有行**，而不是给每组重新标准化。不修改X。非法计数抛 `ValueError`。

演示 `X=np.column_stack((np.arange(20),np.ones(20)))`、n_train=10、n_val=5、seed=20261009，打印索引和统计量；说明测试集用于选择次数会怎样。无外部文件。


<details><summary>提示（先尝试再展开）</summary>

按随机数的排序打乱索引，不需要额外的permutation API。标准差为0的常数列不应除0。

</details>


In [ ]:
import numpy as np
def split_and_scale(X,n_train,n_val,seed):
    n=X.shape[0]
    if n_train<=0 or n_val<=0 or n_train+n_val>=n:
        raise ValueError('all three splits must be nonempty')
    rng=np.random.default_rng(seed)
    order=np.argsort(rng.uniform(0,1,n))
    it=order[:n_train]; iv=order[n_train:n_train+n_val]; ie=order[n_train+n_val:]
    mean=np.mean(X[it],axis=0)
    scale=np.sqrt(np.mean((X[it]-mean)**2,axis=0))
    scale[scale==0]=1
    return dict(train_idx=it,val_idx=iv,test_idx=ie,mean=mean,scale=scale,scaled=(X-mean)/scale)
_X=np.column_stack((np.arange(20),np.ones(20)))
split_result=split_and_scale(_X,10,5,20261009)
for _key in ['train_idx','val_idx','test_idx','mean','scale']:
    print(_key,split_result[_key])
print('Use validation for model selection; reserve testing for final evaluation.')


### 解析与点评

如果改变测试值就改变了预处理均值，说明泄漏。划分是职责分工，不是为了保证每组样本的经验均值完全相等；有限样本存在差异。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
_X=np.column_stack((np.arange(20),np.ones(20))); _copy=_X.copy()
_r=split_and_scale(_X,10,5,20261009)
_it,_iv,_ie=_r['train_idx'],_r['val_idx'],_r['test_idx']
assert len(_it)==10 and len(_iv)==5 and len(_ie)==5
assert set(_it)|set(_iv)|set(_ie)==set(range(20))
assert not(set(_it)&set(_iv) or set(_it)&set(_ie) or set(_iv)&set(_ie))
assert np.allclose(np.mean(_r['scaled'][_it],axis=0),[0,0],atol=1e-12)
assert np.allclose(np.mean(_r['scaled'][_it]**2,axis=0),[1,0],atol=1e-12)
assert _r['scale'][1]==1 and np.array_equal(_X,_copy)
_changed=_X.copy(); _changed[_ie,0]+=10000
_s=split_and_scale(_changed,10,5,20261009)
assert np.allclose(_s['mean'],_r['mean']) and np.allclose(_s['scale'],_r['scale'])
assert np.array_equal(_it,split_and_scale(_X,10,5,20261009)['train_idx'])
try: split_and_scale(_X,15,5,1)
except ValueError: pass
else: raise AssertionError('空测试集应拒绝')

print("自检通过")


## L2-06 容量扫描与过度参数化

**来源与性质**：[[课程/深度学习/PPT/Chapter2/Chapter2.md]]“例1”“容量多大时模型最好”；[[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]] C2-02。改编观察实验。


对L2-04的同一九点数据，实现 `capacity_sweep()`，返回 `(table,coefs)`：table为 `(10,5)`，按p=1—10排列，列为 `[degree,train_rmse,test_rmse,gap_abs,gap_signed]`；coefs字典保存每个p的升幂最小二乘系数。使用 `lstsq(...,rcond=None)`；当列数超过样本数，采用返回的最小范数解。

打印表；绘制训练、测试RMSE随次数变化图，保存 `output/L2-06.png`。解释p>=9时系数不唯一与近零训练误差的关系；指出用本固定测试集反复挑选次数会让其承担验证职责，因此本题仅作描述性观察，不报告“最终泛化性能”。无随机性。


<details><summary>提示（先尝试再展开）</summary>

9个互异样本的8次多项式就可能插值；9、10次是欠定系数问题，但仍可得到一个明确的最小范数返回值。

</details>


In [ ]:
import numpy as np
def poly_matrix(x,p):
    return np.column_stack([x**j for j in range(p+1)])
def poly_ridge(x,y,p,lam):
    if lam < 0:
        raise ValueError('lambda must be nonnegative')
    A=poly_matrix(x,p)
    if lam == 0:
        return np.linalg.lstsq(A,y,rcond=None)[0]
    n=x.size
    B=np.zeros((n+p+1,p+1)); z=np.zeros(n+p+1)
    B[:n,:]=A; z[:n]=y
    for j in range(p+1):
        B[n+j,j]=np.sqrt(n*lam)
    return np.linalg.lstsq(B,z,rcond=None)[0]

import matplotlib.pyplot as plt
from pathlib import Path
def capacity_sweep():
    x=np.array([-.92,-.78,-.34,-.33,-.16,.06,.19,.64,.79]); y=1/(x*x+x+1)
    xt=np.array([-.8,-.4,0.,.4,.8]); yt=1/(xt*xt+xt+1)
    rows,coefs=[],{}
    for p in range(1,11):
        c=np.linalg.lstsq(poly_matrix(x,p),y,rcond=None)[0]; coefs[p]=c
        er=float(np.sqrt(np.mean((poly_matrix(x,p)@c-y)**2)))
        et=float(np.sqrt(np.mean((poly_matrix(xt,p)@c-yt)**2)))
        rows.append([p,er,et,abs(et-er),et-er])
    return np.array(rows),coefs
capacity_table,capacity_coefs=capacity_sweep()
print('degree train_RMSE test_RMSE gap_abs gap_signed',capacity_table,sep='; ')
Path('output').mkdir(exist_ok=True)
plt.figure(); plt.plot(capacity_table[:,0],capacity_table[:,1],'o-',label='train RMSE')
plt.plot(capacity_table[:,0],capacity_table[:,2],'s-',label='test RMSE')
plt.xlabel('degree'); plt.ylabel('RMSE'); plt.legend()
plt.savefig('output/L2-06.png'); plt.close()
print('For degree >= 9 there are more coefficients than training samples; the minimum-norm solution is selected.')
print('Repeatedly choosing by this test table would make it a validation table, not an untouched final test.')


### 解析与点评

训练误差接近0不意味着学习到了整个目标函数。更高次数有更多系数，不保证独立数据上误差更低；不得把本次曲线的细节上升成所有实验的必然规律。这里固定lstsq的最小范数选解，欠定情形的曲线和误差不要求逐位复现课件；本次p=9、10并不呈现课件描述的大幅恶化，不能为了迎合结论篡改数值。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
from pathlib import Path
_t,_c=capacity_sweep()
assert _t.shape==(10,5) and np.array_equal(_t[:,0],np.arange(1,11))
assert np.all(np.diff(_t[:,1])<1e-10) and _t[7,1]<1e-10
assert np.allclose(_t[:,3],np.abs(_t[:,2]-_t[:,1])) and np.allclose(_t[:,4],_t[:,2]-_t[:,1])
_x=np.array([-.92,-.78,-.34,-.33,-.16,.06,.19,.64,.79]); _y=1/(_x*_x+_x+1)
for _p in [1,2,8,9,10]:
    _A=np.column_stack([_x**j for j in range(_p+1)])
    assert _c[_p].shape==(_p+1,) and np.isclose(_t[_p-1,1],np.sqrt(np.mean((_A@_c[_p]-_y)**2)),atol=1e-10)
assert _t[9,2] > _t[8,2]
assert Path('output/L2-06.png').stat().st_size > 1000

print("自检通过")


## L2-07 增广最小二乘实现平方正则化

**来源与性质**：[[课程/深度学习/PPT/Chapter2/Chapter2.md]]“正则化”“例2（1-D正则化模型）”；[[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]] C2-03；[[课程/深度学习/课堂笔记/2026-09-07-第3节-机器学习基础与NumPy实操讲座.md]]“正则化与代码范围”。改编实现。


实现 `poly_ridge(x,y,p,lam)`，返回升幂系数。目标为 `mean((A@w-y)**2)+lam*sum(w**2)`，**惩罚全部系数，包含常数项**，与课件口径一致。lam=0走普通最小二乘；lam<0抛ValueError。

不要显式求逆。构造增广矩阵：上部为A，下部对角元素为 `sqrt(N*lam)`；增广标签下部全0。可用 `np.zeros`、切片与循环赋值完成，不需要表外eye/vstack API。解释N因子的由来。

对九点有理函数比较 p=9,10，各取lam=0与1e-4。打印 `ridge_table` 的4行，列为 `[degree,lambda,train_rmse,test_rmse,coefficient_norm]`，同时将系数存入 `ridge_coeffs[(degree,lambda)]`。lambda是损失中的值，不是直接塞进未归一化正规方程的值。无随机性。


<details><summary>提示（先尝试再展开）</summary>

把平均损失乘N后才对应通常的平方和最小二乘。增广块系数平方应为N*lambda。

</details>


In [ ]:
import numpy as np
def poly_matrix(x,p):
    return np.column_stack([x**j for j in range(p+1)])
def poly_ridge(x,y,p,lam):
    if lam < 0:
        raise ValueError('lambda must be nonnegative')
    A=poly_matrix(x,p)
    if lam == 0:
        return np.linalg.lstsq(A,y,rcond=None)[0]
    n=x.size
    B=np.zeros((n+p+1,p+1)); z=np.zeros(n+p+1)
    B[:n,:]=A; z[:n]=y
    for j in range(p+1):
        B[n+j,j]=np.sqrt(n*lam)
    return np.linalg.lstsq(B,z,rcond=None)[0]
nine_x=np.array([-.92,-.78,-.34,-.33,-.16,.06,.19,.64,.79])
nine_y=1/(nine_x**2+nine_x+1)
nine_xt=np.array([-.8,-.4,0.,.4,.8])
nine_yt=1/(nine_xt**2+nine_xt+1)

ridge_rows=[]; ridge_coeffs={}
for _p in [9,10]:
    for _lam in [0.,1e-4]:
        _c=poly_ridge(nine_x,nine_y,_p,_lam); ridge_coeffs[(_p,_lam)]=_c
        _er=np.sqrt(np.mean((poly_matrix(nine_x,_p)@_c-nine_y)**2))
        _et=np.sqrt(np.mean((poly_matrix(nine_xt,_p)@_c-nine_yt)**2))
        ridge_rows.append([_p,_lam,_er,_et,np.linalg.norm(_c)])
ridge_table=np.array(ridge_rows)
print('degree lambda train_RMSE test_RMSE coefficient_norm',ridge_table,sep='; ')
print('Multiplying the mean-loss objective by N gives ||Aw-y||^2 + N*lambda*||w||^2.')


### 解析与点评

lam=0的欠定问题依旧使用lstsq的最小范数解。ridge的目标比无正则目标多一项，不能直接拿两种目标的标量值当成同一误差。常数项是否惩罚必须声明。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
_x=np.array([-1.,0.,1.]); _y=np.array([2.,1.,6.])
assert np.allclose(poly_ridge(_x,_y,2,0),[1,2,3])
assert np.allclose(poly_ridge(np.array([0.,1.]),np.array([2.,4.]),0,.5),[2.])
try: poly_ridge(_x,_y,2,-.1)
except ValueError: pass
else: raise AssertionError('负正则化系数应拒绝')
assert ridge_table.shape==(4,5)
_x=np.array([-.92,-.78,-.34,-.33,-.16,.06,.19,.64,.79]); _y=1/(_x*_x+_x+1)
for _p in [9,10]:
    _A=np.column_stack([_x**j for j in range(_p+1)])
    _w=ridge_coeffs[(_p,1e-4)]; _w0=ridge_coeffs[(_p,0.)]
    assert np.linalg.norm(_A.T@(_A@_w-_y)+9e-4*_w)<1e-10
    assert np.linalg.norm(_w)<np.linalg.norm(_w0)
    _J=lambda c: np.mean((_A@c-_y)**2)+1e-4*np.sum(c*c)
    assert _J(_w)<=_J(_w0)+1e-12

print("自检通过")


## L2-08 固定模型下改变训练样本数

**来源与性质**：[[课程/深度学习/PPT/Chapter2/Chapter2.md]]“本章Python实践”“数据集大小对误差的影响”；[[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]] C2-04、C2-11；[[课程/深度学习/演示代码/Chapter2.py]]100／20／7次的正弦实验。改编控制变量实验。


实现 `size_experiment(seed=20261011)`：依次从局部rng生成100个 `[-1,1)` 的候选训练横坐标，再生成200个独立测试横坐标；标签均为 `sin(2*pi*x)`，不添加噪声。用候选训练前8、20、100点分别拟合**7次**多项式，固定同一200点测试集。

返回 `(table,coefs,x_pool,x_test)`，table三行列为 `[n_train,train_rmse,test_rmse,gap_abs,gap_signed]`，coefs按n_train作键。打印误差表、画三个学习器与真实函数（训练点可只画100点），保存 `output/L2-08.png`。

说明为何这个实验不能证明“训练样本增加时平均训练误差必然单调增，测试误差必然单调降”；说明原脚本100训练／20测试与本题200测试的区别。比较时只改变训练量；无须反复重设rng而导致数据关系改变。


<details><summary>提示（先尝试再展开）</summary>

训练前缀保证不同样本量之间是嵌套数据，不是三个完全不同的随机问题；测试集在全部模型间固定。

</details>


In [ ]:
import numpy as np
def poly_matrix(x,p):
    return np.column_stack([x**j for j in range(p+1)])
def poly_ridge(x,y,p,lam):
    if lam < 0:
        raise ValueError('lambda must be nonnegative')
    A=poly_matrix(x,p)
    if lam == 0:
        return np.linalg.lstsq(A,y,rcond=None)[0]
    n=x.size
    B=np.zeros((n+p+1,p+1)); z=np.zeros(n+p+1)
    B[:n,:]=A; z[:n]=y
    for j in range(p+1):
        B[n+j,j]=np.sqrt(n*lam)
    return np.linalg.lstsq(B,z,rcond=None)[0]

import matplotlib.pyplot as plt
from pathlib import Path
def size_experiment(seed=20261011):
    rng=np.random.default_rng(seed)
    xp=rng.uniform(-1,1,100); xt=rng.uniform(-1,1,200)
    yp=np.sin(2*np.pi*xp); yt=np.sin(2*np.pi*xt)
    rows,coefs=[],{}
    for n in [8,20,100]:
        A=poly_matrix(xp[:n],7)
        c=np.linalg.lstsq(A,yp[:n],rcond=None)[0]; coefs[n]=c
        er=np.sqrt(np.mean((A@c-yp[:n])**2))
        et=np.sqrt(np.mean((poly_matrix(xt,7)@c-yt)**2))
        rows.append([n,er,et,abs(et-er),et-er])
    return np.array(rows),coefs,xp,xt
size_table,size_coefs,size_pool,size_test=size_experiment()
print('n_train train_RMSE test_RMSE gap_abs gap_signed',size_table,sep='; ')
Path('output').mkdir(exist_ok=True)
plt.figure(); _g=np.linspace(-1,1,301)
plt.plot(_g,np.sin(2*np.pi*_g),'k--',label='target')
for _n,_c in size_coefs.items(): plt.plot(_g,poly_matrix(_g,7)@_c,label=f'n={_n}')
plt.plot(size_pool,np.sin(2*np.pi*size_pool),'.',label='pool')
plt.xlabel('x'); plt.ylabel('y'); plt.legend()
plt.savefig('output/L2-08.png'); plt.close()
print('A single nested-data experiment does not prove monotonic error changes for all samples.')
print('This controlled variant uses 200 test points; the original demo uses 20.')


### 解析与点评

8点、8系数可能插值，却在点外误差很大。源演示没有标签噪声，本题也不凭空给它加噪声；只更改并明确了测试点数与比较组。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
from pathlib import Path
_t,_cs,_xp,_xt=size_experiment()
_rng=np.random.default_rng(20261011)
assert np.array_equal(_xp,_rng.uniform(-1,1,100)) and np.array_equal(_xt,_rng.uniform(-1,1,200))
assert np.array_equal(_t[:,0],[8,20,100]) and _t[0,1]<1e-9
for _i,_n in enumerate([8,20,100]):
    _A=np.column_stack([_xp[:_n]**j for j in range(8)])
    _At=np.column_stack([_xt**j for j in range(8)])
    assert np.isclose(_t[_i,1],np.sqrt(np.mean((_A@_cs[_n]-np.sin(2*np.pi*_xp[:_n]))**2)),atol=1e-10)
    assert np.isclose(_t[_i,2],np.sqrt(np.mean((_At@_cs[_n]-np.sin(2*np.pi*_xt))**2)),atol=1e-10)
assert np.allclose(_t[:,3],np.abs(_t[:,2]-_t[:,1])) and np.allclose(_t[:,4],_t[:,2]-_t[:,1])
assert Path('output/L2-08.png').stat().st_size > 1000

print("自检通过")


## L2-09 训练—验证—测试完整选型实验

**来源与性质**：[[课程/深度学习/PPT/Chapter2/Chapter2.md]]验证集、正则化、正弦演示；[[课程/深度学习/课堂笔记/2026-09-02-第2节-机器学习基础概念与模型构建教学.md]]第八、九节；[[课程/深度学习/课堂笔记/2026-09-07-第3节-机器学习基础与NumPy实操讲座.md]]第七节。综合新编，不是原Chapter2.py已经实现的功能。


实现 `validation_pipeline(seed=20261012)`：rng依次生成120个 `[-1,1)` 横坐标，再生成120个 `[-.08,.08)` 噪声，观测标签 `sin(2*pi*x)+noise`。按生成顺序前60训练、接30验证、后30测试，三组均为独立同分布样本。训练/验证/测试误差均对观测标签计算。

候选为次数 `[3,7,11]` 与lambda `[0,1e-4,1e-2]` 的9种组合；正则目标与L2-07相同，惩罚全部系数。仅用60训练点拟合，以验证RMSE最小者选型；恰好相同时按较小次数、再较小lambda破同分。选完后用前90点重拟合一次，仅最后评价测试集。

返回字典：`candidates`（9行×4列，列为p、lambda、训练RMSE、验证RMSE），`chosen`（二元组），`coef`（最终系数），`train_rmse`（最终90点），`test_rmse`、`gap_abs`、`gap_signed`、`x`、`y`、`test_pred`。

打印候选表和最终指标；画真实正弦、最终模型和三组观测点，保存 `output/L2-09.png`。解释：选择阶段不能用测试RMSE；重拟合后的“训练误差”指90点，不是原60点；单个seed不能证明某正则化系数普遍最好。


<details><summary>提示（先尝试再展开）</summary>

先确定候选表，表中不要出现测试误差。候选的训练行数60决定正则化增广块；最终重拟合行数90会改变N因子。

</details>


In [ ]:
import numpy as np
def poly_matrix(x,p):
    return np.column_stack([x**j for j in range(p+1)])
def poly_ridge(x,y,p,lam):
    if lam < 0:
        raise ValueError('lambda must be nonnegative')
    A=poly_matrix(x,p)
    if lam == 0:
        return np.linalg.lstsq(A,y,rcond=None)[0]
    n=x.size
    B=np.zeros((n+p+1,p+1)); z=np.zeros(n+p+1)
    B[:n,:]=A; z[:n]=y
    for j in range(p+1):
        B[n+j,j]=np.sqrt(n*lam)
    return np.linalg.lstsq(B,z,rcond=None)[0]

import matplotlib.pyplot as plt
from pathlib import Path
def validation_pipeline(seed=20261012):
    rng=np.random.default_rng(seed)
    x=rng.uniform(-1,1,120); y=np.sin(2*np.pi*x)+rng.uniform(-.08,.08,120)
    rows=[]
    for p in [3,7,11]:
        for lam in [0.,1e-4,1e-2]:
            c=poly_ridge(x[:60],y[:60],p,lam)
            er=float(np.sqrt(np.mean((poly_matrix(x[:60],p)@c-y[:60])**2)))
            ev=float(np.sqrt(np.mean((poly_matrix(x[60:90],p)@c-y[60:90])**2)))
            rows.append([p,lam,er,ev])
    best=min(rows,key=lambda row:(row[3],row[0],row[1]))
    p,lam=int(best[0]),best[1]
    c=poly_ridge(x[:90],y[:90],p,lam)
    er=float(np.sqrt(np.mean((poly_matrix(x[:90],p)@c-y[:90])**2)))
    pt=poly_matrix(x[90:],p)@c
    et=float(np.sqrt(np.mean((pt-y[90:])**2)))
    return dict(candidates=np.array(rows),chosen=(p,lam),coef=c,train_rmse=er,
                test_rmse=et,gap_abs=abs(et-er),gap_signed=et-er,x=x,y=y,test_pred=pt)
validation_result=validation_pipeline()
print('degree lambda train60_RMSE validation30_RMSE',validation_result['candidates'],sep='; ')
print('chosen:',validation_result['chosen'])
print('final_train90_RMSE test30_RMSE gap_abs gap_signed')
print(*[validation_result[k] for k in ['train_rmse','test_rmse','gap_abs','gap_signed']])
Path('output').mkdir(exist_ok=True)
plt.figure(); _g=np.linspace(-1,1,301); _p=validation_result['chosen'][0]
plt.plot(_g,np.sin(2*np.pi*_g),'k--',label='target')
plt.plot(_g,poly_matrix(_g,_p)@validation_result['coef'],label='selected and refit')
for _a,_b,_name in [(0,60,'train'),(60,90,'validation'),(90,120,'test')]:
    plt.plot(validation_result['x'][_a:_b],validation_result['y'][_a:_b],'.',label=_name)
plt.xlabel('x'); plt.ylabel('y'); plt.legend(); plt.savefig('output/L2-09.png'); plt.close()
print('Selection uses validation only; the test set is evaluated after the final refit.')
print('Final training RMSE uses all 90 refit points; one seed does not establish a universally best lambda.')


### 解析与点评

这是一条完整的离线建模流程，但不是声称原教师脚本已实现验证、噪声和正则化。数据划分、正则归一化、重拟合样本数与指标名称应一起检查。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
from pathlib import Path
_r=validation_pipeline(); _t=_r['candidates']; _p,_lam=_r['chosen']
assert _t.shape==(9,4)
assert set(map(tuple,_t[:,:2]))=={(p,l) for p in [3,7,11] for l in [0.,1e-4,1e-2]}
_best=min(_t.tolist(),key=lambda r:(r[3],r[0],r[1]))
assert (_p,_lam)==(int(_best[0]),_best[1])
_rng=np.random.default_rng(20261012); _x=_rng.uniform(-1,1,120)
_y=np.sin(2*np.pi*_x)+_rng.uniform(-.08,.08,120)
assert np.allclose(_r['x'],_x) and np.allclose(_r['y'],_y)
_A=np.column_stack([_x[:90]**j for j in range(_p+1)])
assert np.linalg.norm(_A.T@(_A@_r['coef']-_y[:90])+90*_lam*_r['coef'])<1e-8
assert np.isclose(_r['train_rmse'],np.sqrt(np.mean((_A@_r['coef']-_y[:90])**2)))
assert np.isclose(_r['test_rmse'],np.sqrt(np.mean((_r['test_pred']-_y[90:])**2)))
assert np.isclose(_r['gap_abs'],abs(_r['test_rmse']-_r['train_rmse']))
assert np.isclose(_r['gap_signed'],_r['test_rmse']-_r['train_rmse'])
assert _r['test_rmse']<.12
assert Path('output/L2-09.png').stat().st_size > 1000

print("自检通过")
